# Introdução à biblioteca pandas

**Minicurso de Python e pandas — material revisado para estudo e portfólio.**

Material do minicurso organizado e ministrado por Gabriel Bitencourt de Souza, com participação da equipe na preparação do material didático, na Semana Acadêmica de Engenharia de Produção da UFV.

Esta edição revisa o notebook utilizado na apresentação em Colab. Mantém os temas e três exercícios principais, corrige a API e a interpretação dos dados e acrescenta exemplos executáveis e gabarito. As bases desta edição são **inteiramente sintéticas**, criadas para ensino; não representam pessoas, salários, vendas, clubes ou resultados reais. As bases originais do Drive não foram republicadas.

Revisão: 26/10/2026. A data da apresentação não foi inferida a partir dos arquivos.

## Como usar

1. No Colab: conecte-se a um ambiente Python e execute as células em ordem. Não é necessário montar o Drive, autorizar contas ou enviar bases pessoais.
2. Localmente: instale `requirements.txt` e abra o notebook com Jupyter. Execute a partir da primeira célula.
3. A preparação cria/recria apenas as pastas `minicurso_pandas_dados/` e `minicurso_pandas_saidas/` no diretório atual. Não coloque seus arquivos nessas pastas de exemplo.
4. Os blocos em Markdown são referências de sintaxe; as células de código contêm exemplos e espaço para exercícios. O gabarito comentado está em outro notebook.

**Pré-requisitos:** variáveis, listas, dicionários e comparações básicas em Python. pandas trabalha principalmente em memória; arquivos muito grandes podem exigir leitura em partes ou outras ferramentas.


In [ ]:
from pathlib import Path
import csv
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Somente dados sintéticos. Não monta o Drive, não lê arquivos pessoais e não usa rede.
BASE = Path('minicurso_pandas_dados')
SAIDAS = Path('minicurso_pandas_saidas')
BASE.mkdir(exist_ok=True)
SAIDAS.mkdir(exist_ok=True)

TABELAS = {
    'vendas.csv': (
        ['id_venda', 'data_venda', 'nome_produto', 'quantidade', 'valor_unitario', 'valor_total'],
        [[1, '2025-01-02', 'Caderno fictício', 2, 12.5, 25.0],
         [2, '2025-01-03', 'Caneta fictícia', 5, 3.0, 15.0],
         [3, '2025-01-04', 'Caderno fictício', 1, 12.5, 12.5],
         [4, '2025-01-05', 'Pasta fictícia', 0, 8.0, 0.0],
         [5, '2025-01-06', 'Pasta fictícia', 3, 8.0, 24.0],
         [6, '2025-01-07', 'Caneta fictícia', 4, 3.0, 12.0]]),
    'player_stats.csv': (
        ['name', 'team', 'age', 'played', 'goals'],
        [['Atleta fictício A', 'Clube fictício Azul', 24, 12, 0],
         ['Atleta fictício B', 'Clube fictício Verde', 22, 0, 0],
         ['Atleta fictício C', 'Clube fictício Azul', '', 8, 2],
         ['Atleta fictício D', 'Clube fictício Verde', 28, 10, 5]]),
    'player_salaries.csv': (
        ['Player', 'Nation', 'Position', 'Team', 'Age', 'Annual'],
        [['Atleta fictício A', 'BRA', 'FW', 'Clube fictício Azul', 24, 120000],
         ['Atleta fictício B', 'BRA', 'MF', 'Clube fictício Verde', 22, 180000],
         ['Atleta fictício B', 'BRA', 'MF', 'Clube fictício Verde', 22, 180000],
         ['Atleta fictício C', 'BRA', '', 'Clube fictício Azul', 23, 200000],
         ['Atleta fictício D', 'BRA', 'DF', 'Clube fictício Verde', '', 250000],
         ['Atleta fictício E', 'ARG', 'FW', 'Clube fictício Azul', 21, 300000],
         ['Atleta fictício F', 'BRA', 'MF', 'Clube fictício Verde', 27, 400000],
         ['Atleta fictício G', 'BRA', 'GK', 'Clube fictício Azul', 20, '']]),
    'dados_funcionarios.csv': (
        ['ID', 'Nome', 'Departamento', 'Idade', 'Salario', 'Anos_Experiencia'],
        [[1, 'Pessoa fictícia A', 'RH', 28, 3200, 5],
         [2, 'Pessoa fictícia B', 'TI', 34, 5000, 8],
         [3, 'Pessoa fictícia C', 'TI', '', 6000, 10],
         [4, 'Pessoa fictícia D', 'Operações', 24, 2800, 2],
         [5, 'Pessoa fictícia E', 'RH', 31, '', 6],
         [6, 'Pessoa fictícia F', 'Operações', 40, 4500, 15],
         [6, 'Pessoa fictícia F', 'Operações', 40, 4500, 15]])
}
for nome, (colunas, linhas) in TABELAS.items():
    with (BASE / nome).open('w', encoding='utf-8', newline='') as arquivo:
        writer = csv.writer(arquivo)
        writer.writerow(colunas)
        writer.writerows(linhas)

df_vendas = pd.read_csv(BASE / 'vendas.csv', parse_dates=['data_venda'])
# Arquivo de treino para a seção de leitura de Excel, criado só com a base sintética.
df_vendas.to_excel(BASE / 'vendas.xlsx', index=False, engine='openpyxl')
print(f'pandas {pd.__version__} | exemplos sintéticos preparados')


## O que é pandas?
---

pandas é uma biblioteca Python para análise e manipulação de dados tabulares. Uma `Series` é uma estrutura unidimensional com rótulos; um `DataFrame` é uma tabela bidimensional cujas colunas podem ter tipos diferentes. NumPy é uma dependência importante, mas pandas também oferece tipos e comportamentos próprios, como dados anuláveis e alinhamento por rótulos.

## Por que usar pandas?
---

O pandas é amplamente adotado por cientistas de dados e analistas porque facilita diversas tarefas, como:

* Limpeza de dados: Tratamento de valores ausentes ou inconsistentes.
* Preparação de dados: Transformação e organização para análise.
* Análise: Operações como filtragem, agrupamento e cálculos estatísticos.
* Visualização: Integração com outras bibliotecas para gráficos.

A escolha depende do volume e da memória disponível. Neste minicurso usamos bases pequenas para entender cada transformação.


In [ ]:
print(f'Versão do pandas: {pd.__version__}')

# Series e DataFrames 📊

## Series
---

Uma Series é como uma lista ou array de uma dimensão, mas com um diferencial importante: cada valor tem um rótulo (índice). Isso permite que você acesse os elementos por um nome, não só por posição.

```python
import pandas as pd

s = pd.Series([1, 2, 3], index=['a', 'b', 'c'])
print(s)
```
O que está acontecendo aqui:

- Criamos uma Series com três valores: 1, 2 e 3.

- A lista `['a', 'b', 'c']` define os índices personalizados desses valores.

O resultado vai ser algo assim:


```
a    1
b    2
c    3
dtype: int64
```
Ou seja, agora podemos acessar `s.loc['b']` e obter o valor 2, por exemplo.

#### Exercício:
Crie uma série e retorne um de seus valores pelo índice.

Use `s.loc['b']` para rótulos e `s.iloc[1]` para posições. Rótulos não precisam ser únicos; com duplicatas, uma seleção pode retornar mais de um valor.

In [ ]:
s = pd.Series([1, 2, 3], index=['a', 'b', 'c'])
print(s)
print('Rótulo b:', s.loc['b'], '| Posição 1:', s.iloc[1])
# Exercício: crie outra Series e selecione um valor.

## DataFrames
---

Enquanto a Series é unidimensional, o DataFrame é bidimensional – como uma tabela, com linhas e colunas. É a estrutura central do Pandas.

Exemplo:
```python
df = pd.DataFrame({
    'A': [1, 2, 3],
    'B': ['x', 'y', 'z']
})
print(df)

```
- Criamos um dicionário onde cada chave ('A' e 'B') representa uma coluna.

- Cada lista fornece os valores de uma coluna; as posições correspondentes formam as linhas.

O resultado será:
```
   A  B
0  1  x
1  2  y
2  3  z
```
Também é possível atribuir valores aos índices de um DataFrame, como no exemplo abaixo:
```python
df = pd.DataFrame({
    'A': [1, 2, 3],
    'B': ['x', 'y', 'z']
}, index=['linha1', 'linha2', 'linha3'])
```
A saída, nesse caso, será:
```
         A  B
linha1   1  x
linha2   2  y
linha3   3  z
```
Uma vez que você tem um DataFrame, pode inspecionar suas propriedades usando alguns atributos. Vamos usar o mesmo DataFrame df:

| Atributo   | Descrição                                                   |
| ---------- | ----------------------------------------------------------- |
| `.columns` | Retorna os nomes das colunas                                |
| `.index`   | Retorna os índices das linhas                               |
| `.dtypes`  | Tipo de dado de cada coluna                                 |
| `.shape`   | Tupla (n\_linhas, n\_colunas)                               |
| `.size`    | Número total de elementos                                   |
| `.ndim`    | Número de dimensões (normalmente 2)                         |
| `.values`  | Array NumPy; prefira o método `.to_numpy()`                     |
| `.T`       | Transposta do DataFrame (linhas viram colunas e vice-versa) |
| `.empty`   | `True` se algum eixo tiver comprimento zero                 |


#### Exercício:

Crie um DataFrame e retorne sua estrutura, a quantidade de linhas e colunas, seus índices e suas colunas. Tente criar um com os índices padrão, e outro personalizando os índices. Compare os resultados.

Um DataFrame preenchido somente com valores ausentes não é considerado vazio por `.empty`. Seu `.ndim` é sempre 2.

In [ ]:
df = pd.DataFrame({'A': [1, 2, 3], 'B': ['x', 'y', 'z']})
display(df)
print('Forma:', df.shape, '| Tamanho:', df.size, '| Dimensões:', df.ndim)
print('Índice:', df.index.tolist(), '| Colunas:', df.columns.tolist())
# Exercício: personalize o índice e compare.

# Leitura e escrita de dados 📖🖊️

## Leitura de dados
---

### Lendo Arquivos CSV

Arquivos CSV (Comma-Separated Values) são muito usados para armazenar dados em formato de tabela. No Python, a biblioteca pandas facilita muito a leitura desses arquivos.

No Google Colab, o processo funciona assim:

Fazemos o upload do arquivo CSV para o ambiente do Colab.

Usamos `pd.read_csv()` para carregar os dados em um DataFrame, que é uma estrutura de tabela que permite manipular e analisar os dados facilmente.

```python
# Lendo um arquivo CSV enviado para o Colab
df = pd.read_csv(BASE / 'vendas.csv')

# Mostra as 5 primeiras linhas da tabela
df.head()
```
A tabela a seguir mostra os principais argumentos para a função `read_csv`.

| Argumento            | Função                                   | Exemplo de uso                                  |
| -------------------- | ---------------------------------------- | ----------------------------------------------- |
| `filepath_or_buffer` | Caminho do arquivo                       | `"dados.csv"`                                   |
| `sep`                | Separador de campos                      | `sep=';'` (depende do formato do arquivo)              |
| `header`             | Linha usada como cabeçalho               | `header=0` (primeira linha) ou `header=None`    |
| `names`              | Atribui nomes; `header=0` substitui cabeçalhos existentes | `names=['produto', 'valor', 'data']`            |
| `index_col`          | Coluna usada como índice                 | `index_col='id'` ou `index_col=0`               |
| `usecols`            | Colunas a importar                       | `usecols=['nome', 'preço']` ou `usecols=[0, 2]` |
| `dtype`              | Tipo de dado por coluna                  | `dtype={'quantidade': int, 'preco': float}`     |
| `parse_dates`        | Converte coluna(s) para datetime         | `parse_dates=['data_venda']`                    |
| `encoding`           | Codificação do arquivo                   | `encoding='utf-8'` ou `encoding='latin1'`       |
| `na_values`          | Valores tratados como NaN                | `na_values=['NA', 'null', '-']`                 |

#### Exercício

Tente importar o arquivo vendas.csv! Teste alguns argumentos e observe o comportamento do dataframe resultante.

In [ ]:
df_vendas = pd.read_csv(BASE / 'vendas.csv', parse_dates=['data_venda'])
display(df_vendas.head())
# Exercício: experimente usecols e confira o tipo das datas.

### Lendo Arquivos Excel

Além de CSV, o pandas também permite ler arquivos Excel (.xlsx), que são muito comuns em relatórios e planilhas do dia a dia.

Assim como no CSV, no Colab precisamos fazer o upload do arquivo primeiro. Depois usamos a função `pd.read_excel()`.

```python
# Lendo um arquivo Excel enviado
df_excel = pd.read_excel(BASE / 'vendas.xlsx', engine='openpyxl')

df_excel.head()
```

A tabela a seguir mostra os principais argumentos para a função `read_excel`.

| Argumento     | Função                                        | Exemplo de uso                                    |
| ------------- | --------------------------------------------- | ------------------------------------------------- |
| `io`          | Caminho do arquivo Excel                      | `"vendas.xlsx"`                                   |
| `sheet_name`  | Aba a ser lida (`int`, `str`, `list`, `None`) | `sheet_name='Janeiro'` ou `sheet_name=0`          |
| `header`      | Linha usada como cabeçalho                    | `header=0` ou `header=None`                       |
| `names`       | Atribui nomes; `header=0` substitui cabeçalhos existentes      | `names=['produto', 'valor', 'data']`              |
| `index_col`   | Coluna usada como índice                      | `index_col='id'`                                  |
| `usecols`     | Quais colunas importar                        | `usecols="A:C"` ou `usecols=['produto', 'valor']` |
| `dtype`       | Tipo de dado por coluna                       | `dtype={'valor': float}`                          |
| `parse_dates` | Converte coluna(s) para datetime              | `parse_dates=['data_venda']`                      |
| `na_values`   | Valores tratados como NaN                     | `na_values=['vazio', '-']`                        |
| `engine`      | Motor de leitura                              | `engine='openpyxl'` (para `.xlsx`; exige a dependência)              |

#### Exercício

Tente importar o arquivo vendas.xlsx! Teste alguns argumentos e observe o comportamento do dataframe resultante.

In [ ]:
df_excel = pd.read_excel(BASE / 'vendas.xlsx', engine='openpyxl')
display(df_excel.head())
# Exercício: teste sheet_name e usecols.

## Escrita de dados
---

### Escrita de dados em arquivos CSV

Depois de manipular ou criar dados no pandas, você pode salvar esses dados em um arquivo CSV para usar fora do ambiente Python, compartilhar ou guardar para análises futuras.

Podemos salvar um DataFrame como CSV usando o método `to_csv()` do pandas.

```python
# Salvando o DataFrame em um arquivo CSV
df_vendas.to_csv(SAIDAS / 'vendas_salvas.csv', index=False, encoding='utf-8')

# No Colab, baixe pelo painel Arquivos ou, opcionalmente:
# from google.colab import files
# files.download(str(SAIDAS / 'vendas_salvas.csv'))
```

A tabela a seguir mostra os principais argumentos para o método `to_csv`.

| Argumento         | Função                                              | Exemplo de uso                                   |
| ----------------- | --------------------------------------------------- | ------------------------------------------------ |
| `path_or_buf`     | Caminho ou buffer onde salvar o CSV                 | `'dados.csv'`                                    |
| `sep`             | Separador de campos                                 | `sep=';'`                                        |
| `index`           | Salvar índice junto?                                | `index=False` (remove o índice do CSV)           |
| `columns`         | Quais colunas salvar                                | `columns=['produto', 'preco']`                   |
| `header`          | Incluir cabeçalho?                                  | `header=True`                                    |
| `encoding`        | Codificação do arquivo                              | `encoding='utf-8'` ou `'latin1'`                 |
| `na_rep`          | Representação de valores ausentes                   | `na_rep='NULL'` ou `na_rep='-'`                  |
| `decimal`         | Símbolo decimal (útil para países que usam vírgula) | `decimal=','`                                    |
| `lineterminator` | Delimitador de linha                                | `lineterminator='\n'` ou `'\r\n'`               |
| `float_format`    | Formatação de números decimais                      | `float_format='%.2f'`                            |
| `date_format`     | Formato para colunas de data                        | `date_format='%Y-%m-%d'`                         |
| `mode`            | Modo de abertura do arquivo                         | `mode='w'` (escrita) ou `mode='a'` (acréscimo)   |
| `compression`     | Compressão do arquivo                               | `compression='gzip'`, `'zip'`, `'bz2'` ou `None` |

#### Exercício:

Crie um DataFrame, depois tente salvá-lo em um arquivo .csv.


In [ ]:
df_vendas.to_csv(SAIDAS / 'vendas_salvas.csv', index=False, encoding='utf-8', lineterminator='\n')
print('CSV sintético salvo na pasta de saídas.')
# Exercício: leia o arquivo exportado e confira as colunas.

### Escrita de dados em arquivos Excel

Também podemos exportar para Excel. O arquivo abaixo contém somente dados sintéticos:

```python
df_vendas.to_excel(SAIDAS / 'vendas_salvas.xlsx', index=False, engine='openpyxl')
```

| Argumento de `to_excel` | Função | Exemplo |
|---|---|---|
| `excel_writer` | Caminho ou objeto `ExcelWriter` | `SAIDAS / 'vendas_salvas.xlsx'` |
| `sheet_name` | Nome da aba | `'Vendas'` |
| `index` | Incluir rótulos do índice no arquivo | `False` |
| `columns` | Selecionar colunas | `['nome_produto', 'valor_total']` |
| `header` | Incluir cabeçalho | `True` |
| `na_rep` | Representação da ausência na exportação | `'NA'` |
| `float_format` | Formatação de números na escrita | `'%.2f'` |
| `startrow`, `startcol` | Posição inicial, a partir de zero | `startrow=2` |
| `engine` | Motor de escrita disponível | `'openpyxl'` |

`date_format`, `datetime_format` e `mode` pertencem a **`pd.ExcelWriter`**, não diretamente a `DataFrame.to_excel`. Para acrescentar uma aba a um arquivo de exemplo já existente:

```python
with pd.ExcelWriter(SAIDAS / 'vendas_salvas.xlsx', engine='openpyxl', mode='a',
                    if_sheet_exists='replace', datetime_format='DD/MM/YYYY') as writer:
    df_vendas.to_excel(writer, sheet_name='Vendas', index=False)
```

`engine_kwargs` depende do motor. A opção `strings_to_formulas` é específica do XlsxWriter e não deve ser passada ao openpyxl. A escrita padrão pode substituir arquivos; utilize somente os destinos de demonstração. No Colab, faça o download pelo painel Arquivos ou importe `files` de `google.colab` explicitamente.

#### Exercício
Exporte os dados sintéticos para um CSV e um Excel. Leia ambos novamente e compare as colunas e o número de registros.


In [ ]:
df_vendas.to_excel(SAIDAS / 'vendas_salvas.xlsx', index=False, engine='openpyxl')
print('Excel sintético salvo na pasta de saídas.')
# Exercício: leia novamente o arquivo e compare com o DataFrame.

# Exercício 1: Começando com Pandas 🚀

Pratique importação e exploração com `player_stats.csv`, que nesta edição representa uma liga **inteiramente fictícia**.

1. Leia apenas `name`, `team`, `age`, `played` e `goals`.
2. Preserve os zeros de partidas e gols: zero é um resultado válido, não ausência. Campos vazios já representam dados ausentes. Não use `na_values=0` globalmente.
3. Exiba as primeiras 5, 2 e 4 linhas com `head`. Quando houver menos de 5 registros, `head(5)` retorna os disponíveis.
4. Inspecione `dtypes`, valores ausentes e estatísticas descritivas numéricas.
5. Exporte para `SAIDAS / 'player_stats_alterados.csv'`, sem índice e com ausência representada por `NA`. Essa representação na exportação não preenche a idade ausente no DataFrame.
6. Explique por que transformar um número de gols igual a zero em `NaN` mudaria a média. Em que situação um código sentinela poderia ser tratado como ausência? Somente com uma regra documentada para a coluna.


In [ ]:
# Exercício 1: escreva aqui. A base está em BASE / 'player_stats.csv'.
# O gabarito está no notebook WorkshopPandas_gabarito.ipynb.

# Manipulação de dados 🎲

Manipular dados significa extrair e organizar as informações que realmente
importam para sua análise.
Com o pandas, isso inclui operações como selecionar colunas, filtrar linhas, ordenar registros e alterar o índice.

Vamos usar um DataFrame simples como exemplo:

```python
import pandas as pd

df = pd.DataFrame({
    'nome': ['Ana', 'Bruno', 'Carlos', 'Daniela'],
    'idade': [22, 35, 28, 42],
    'salario': [3000, 5000, 4000, 5500]
})

print(df)
```
O que está acontecendo aqui:

*   Criamos um DataFrame com 4 pessoas.

*   Cada pessoa tem um nome, idade e salário.


A saída seria:

```
     nome  idade  salario
0     Ana     22     3000
1   Bruno     35     5000
2  Carlos     28     4000
3 Daniela     42     5500
```

## Seleção de Colunas
---

Para acessar apenas uma coluna deve-se usar:

```python
df['idade']
```

Aqui estamos acessando a coluna "idade", que retorna uma Series.

A saída seria:

```
0    22
1    35
2    28
3    42
Name: idade, dtype: int64
```
Já agora, selecionamos duas colunas ao mesmo tempo:
```python
df[['nome', 'salario']]
```
Isso retorna um novo DataFrame com apenas essas colunas.

A saída seria:
```
     nome  salario
0     Ana     3000
1   Bruno     5000
2  Carlos     4000
3 Daniela     5500
```

Note que estamos filtrando um banco de dados por suas colunas. A filtragem pode ser feita através de uma string, que retornará uma Series, ou através de uma **lista** de strings, que retornará um DataFrame. De modo geral:

| Expressão              | Tipo retornado | Explicação                           |
| ---------------------- | -------------- | ------------------------------------ |
| `df['coluna']`         | `Series`       | Uma coluna como vetor unidimensional |
| `df[['coluna']]`       | `DataFrame`    | Uma coluna como tabela (ainda 2D)    |
| `df[['col1', 'col2']]` | `DataFrame`    | Várias colunas                       |

#### Exercício:

Utilizando o DataFrame do início deste módulo, filtre-o por sua coluna `nome`, retornando um DataFrame como resultado. Tente fazer o mesmo filtrando duas colunas simultaneamente.

In [ ]:
df = pd.DataFrame({'nome': ['Ana', 'Bruno', 'Carlos', 'Daniela'],
                   'idade': [22, 35, 28, 42], 'salario': [3000, 5000, 4000, 5500]})
display(df[['nome']])
# Exercício: selecione duas colunas.

## Seleção de Linhas
---

Existem dois modos de acessar dados de linhas de um DataFrame: utilizando os *indexers* `loc[]` e `iloc[]`.

O `.loc` permite acessar elementos de um DataFrame usando os nomes (ou rótulos) do índice e das colunas. É ideal quando você quer trabalhar com dados de forma mais explícita e legível.


```python
df.loc[3, 'nome']
```
Esse comando retorna o valor da coluna 'nome' na linha de índice 3.

O `.iloc` faz a mesma coisa que o `.loc`, mas usa posições inteiras a partir de zero (como em listas Python). Isso significa que ele acessa os dados com base na posição física das linhas e colunas, e não nos nomes.

```python
df.iloc[3, 1]
```

Esse comando retorna o valor da quarta linha (posição 3) e da segunda coluna (posição 1).

A tabela abaixo resume as formas de utilizar o `.loc[]` e o `.iloc[]`.

| Método   | Argumentos (forma)                        | Exemplo                                               |
| -------- | ----------------------------------------- | ----------------------------------------------------- |
| `loc[]`  | `df.loc[linha, coluna]`                   | `df.loc[2, 'nome']` – valor na linha 2, coluna 'nome' |
|          | `df.loc[fatia_linha, lista_colunas]`      | `df.loc[1:3, ['nome', 'idade']]`                      |
|          | `df.loc[condição]`                        | `df.loc[df['idade'] > 30]`                            |
| `iloc[]` | `df.iloc[linha, coluna]`                  | `df.iloc[0, 1]` – valor da 1ª linha e 2ª coluna       |
|          | `df.iloc[início:fim, col_início:col_fim]` | `df.iloc[0:2, 0:2]`                                   |
|          | `df.iloc[[0, 2], [1, 2]]`                 | Linhas 0 e 2, colunas 1 e 2                           |

#### Exercício:

Tente utilizar o `loc` ou o `iloc` para filtrar linhas do DataFrame do início deste módulo.

Na fatia `df.loc[1:3]`, os rótulos 1 e 3 são incluídos quando estão presentes. Em `df.iloc[1:3]`, a posição 3 fica de fora. Isso é diferente mesmo quando os rótulos são números.

In [ ]:
display(df.loc[1:3, ['nome', 'idade']])
display(df.iloc[1:3, [0, 1]])
# Exercício: compare o número de linhas das duas seleções.

## Filtragem com Condições
---

O Pandas permite filtrar linhas usando expressões condicionais, que retornam uma máscara booleana (True/False) aplicada ao DataFrame.

Exemplo básico:

```python
df[df['idade'] > 30]  # retorna apenas as linhas com idade > 30
```

Ou de forma mais explícita:

```python
df.loc[df['idade'] > 30]
```
Os operadores de comparação utilizados para a filtragem estão dispostos na tabela abaixo.

| Operador | Significado    | Exemplo                | Significado lógico     |
| -------- | -------------- | ---------------------- | ---------------------- |
| `==`     | Igual          | `df['idade'] == 30`    | Idade exatamente 30    |
| `!=`     | Diferente      | `df['sexo'] != 'M'`    | Sexo não é masculino   |
| `>`      | Maior          | `df['idade'] > 25`     | Idade acima de 25      |
| `<`      | Menor          | `df['salario'] < 5000` | Salário abaixo de 5000 |
| `>=`     | Maior ou igual | `df['nota'] >= 7`      | Nota 7 ou mais         |
| `<=`     | Menor ou igual | `df['nota'] <= 5`      | Nota 5 ou menos        |


Já os operadores lógicos são os seguintes:

| Operador | Significado | Exemplo |
|---|---|---|
| `&`      | E (AND)     | `(df['idade'] > 30) & (df['sexo'] == 'F')`            |
| `\|`       |   OU (OR) | `(df['cidade'] == 'SP') \| (df['cidade'] == 'RJ')`                                               |
| `~`      | NÃO (NOT)   | `~(df['ativo'])` — seleciona onde `ativo` for `False` |

> ⚠️ Note que é importante utilizar parênteses em volta de cada condição ao utilizar os operadores lógicos acima.

#### Exercício:

Utilize o DataFrame a seguir para filtrar apenas as pessoas do sexo masculino, com salário maior que R$5000,00 e que são de RJ.


In [ ]:
import pandas as pd

df = pd.DataFrame({
    'nome': ['Ana', 'Bruno', 'Carlos', 'Daniela', 'Eduardo', 'Fernanda'],
    'idade': [28, 35, 22, 41, 30, 33],
    'sexo': ['F', 'M', 'M', 'F', 'M', 'F'],
    'cidade': ['SP', 'RJ', 'SP', 'MG', 'RJ', 'SP'],
    'salario': [4800, 5500, 3000, 7000, 5200, 6000],
    'nota': [8.5, 6.0, 9.0, 5.5, 7.0, 4.0],
    'ativo': [True, True, False, True, False, True]
})
# Exercício: filtre M, salário acima de 5000 e cidade RJ usando & e parênteses.

## Ordenação
---

O método `sort_values()` permite ordenar um DataFrame com base nos valores de uma ou mais colunas.

```python
df.sort_values(by='coluna', ascending=True)
```
Os parâmetros são os seguintes:

| Parâmetro     | Função                                                                   |
| ------------- | ------------------------------------------------------------------------ |
| `by`          | Nome da coluna (ou lista de colunas) pela qual ordenar                   |
| `ascending`   | `True` (crescente, padrão) ou `False` (decrescente)                      |
| `inplace`     | `True` modifica o DataFrame original; `False` retorna uma cópia ordenada |
| `na_position` | `'first'` ou `'last'`: posição dos valores ausentes (`NaN`)              |

Exemplos práticos:

```python
# 1. Ordenar por idade (crescente):
df.sort_values(by='idade')

# 2. Ordenar por salário (decrescente):
df.sort_values(by='salario', ascending=False)

# 3. Ordenar por múltiplas colunas (nota decrescente, depois idade crescente):
df.sort_values(by=['nota', 'idade'], ascending=[False, True])

#4. Ordenar e aplicar a alteração diretamente ao DataFrame:
df.sort_values(by='nome', inplace=True)
```

#### Exercício:

Ordene o DataFrame abaixo por salário, em ordem decrescente.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    'nome': ['Ana', 'Bruno', 'Carlos', 'Daniela', 'Eduardo', 'Fernanda'],
    'idade': [28, 35, 22, 41, 30, 33],
    'sexo': ['F', 'M', 'M', 'F', 'M', 'F'],
    'cidade': ['SP', 'RJ', 'SP', 'MG', 'RJ', 'SP'],
    'salario': [4800, 5500, 3000, 7000, 5200, 6000],
    'nota': [8.5, 6.0, 9.0, 5.5, 7.0, 4.0],
    'ativo': [True, True, False, True, False, True]
})
# Exercício: ordene o salário em ordem decrescente, sem modificar df.

## Indexação
---

Podemos mudar o índice do DataFrame para facilitar buscas:
```python
df.set_index('nome', inplace=True)
```

A coluna nome vira o novo índice do DataFrame, substituindo os números padrão.


Podemos obter novamente o índice numérico padrão, com a coluna nome voltando a ser uma coluna comum do DataFrame, utilizando o seguinte comando:

```python
df.reset_index(inplace=True)
```
#### Exercício:

Selecione a coluna `nome` como índice para o DataFrame fornecido. Tente localizar com `loc` os dados referentes à Ana. Após isso, reverta o processo, retornando aos índices numéricos.

Mudar o índice não torna os rótulos únicos automaticamente. Antes de usar um nome como identificador, verifique se ele realmente identifica uma pessoa ou registro.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    'nome': ['Ana', 'Bruno', 'Carlos', 'Daniela'],
    'idade': [22, 35, 28, 42],
    'salario': [3000, 5000, 4000, 5500]
})
# Exercício: use nome como índice, localize Ana e restaure o índice.

# Limpeza de Dados 🧹

Limpar dados é uma etapa essencial antes de qualquer análise. É comum encontrarmos valores ausentes, duplicados ou colunas com tipos incorretos. Vamos ver como o pandas pode nos ajudar com tudo isso.

Exemplo:
```python
import pandas as pd

df = pd.DataFrame({
    'nome': ['Ana', 'Bruno', 'Carlos', 'Ana', 'Eduarda'],
    'idade': [25, 30, None, 25, 29],
    'salario': [3000, 4000, 3500, 3000, None],
    'data_admissao': ['2020-01-01', '2019-05-20', '2021-03-15', '2020-01-01', '2022-07-10']
})

print(df)

```
Esse DataFrame está estruturado da seguinte forma:
```
     nome  idade  salario data_admissao
0     Ana   25.0   3000.0    2020-01-01
1   Bruno   30.0   4000.0    2019-05-20
2  Carlos    NaN   3500.0    2021-03-15
3     Ana   25.0   3000.0    2020-01-01
4 Eduarda   29.0      NaN    2022-07-10
```

Acabamos de criar um DataFrame com alguns problemas comuns e recorrentes em banco de dados:

*   A idade de Carlos está ausente.
*   O salário da Eduarda está ausente.
*   Ana aparece duas vezes.
*   A coluna de datas está como string (texto), e não como data.

É necessário tratar esses dados, de forma a manter o banco de dados adequado a análises.

São pessoas fictícias. Neste exemplo a linha inteira de Ana foi repetida intencionalmente. Em uma base real, nomes iguais não bastam para concluir que há duplicata. Uma data em texto deve ser convertida antes de cálculos temporais.

## Valores Ausentes
---

Para identificar valores ausentes, utilizamos o método `isnull()`:
```python
df.isnull()
```
Para o DataFrame que estamos estudando, a saída será:
```
    nome  idade  salario  data_admissao
0  False  False    False          False
1  False  False    False          False
2  False   True    False          False
3  False  False    False          False
4  False  False     True          False
```
Para saber quantos valores ausentes há em cada coluna, podemos somar todas as ocorrências de valores ausentes:
```python
df.isnull().sum()
```
O resultado será uma Series no seguinte formato:
```
nome             0
idade            1
salario          1
data_admissao    0
dtype: int64
```
Para o tratamento, utilizaremos o método `fillna()`. Primeiro, iremos preencher a idade faltante com a média das idades:
```python
df['idade'] = df['idade'].fillna(df['idade'].mean())
```
Depois, preencheremos o salário faltante com zero:
```python
df['salario'] = df['salario'].fillna(0)
```
A saída será:

```
     nome      idade  salario data_admissao
0     Ana  25.000000   3000.0    2020-01-01
1   Bruno  30.000000   4000.0    2019-05-20
2  Carlos  27.25      3500.0    2021-03-15
3     Ana  25.000000   3000.0    2020-01-01
4 Eduarda  29.000000      0.0    2022-07-10
```
Também é possível remover linhas com valores ausentes, utilizando o método `dropna`:
```python
df.dropna(inplace=True)
```

#### Exercício:

Repita o código estruturado acima, observando o resultado.

**Interpretação:** média e zero são escolhas didáticas, não regras universais. Salário ausente não significa salário zero. Antes de imputar, registre a justificativa e o efeito nos resultados. A média 27,25 inclui a linha de Ana repetida; removê-la antes do cálculo produziria 28,0. A ordem do tratamento importa.

O padrão `df['coluna'].fillna(..., inplace=True)` não deve ser usado para atualizar o DataFrame original sob Copy-on-Write. Prefira atribuição explícita.

`dropna()` remove conforme o critério escolhido. Evite eliminar uma linha porque uma coluna irrelevante está vazia. Depois de preencher todas as ausências desse exemplo, `df.dropna()` não removeria nenhuma linha.

Para converter datas:
```python
df['data_admissao'] = pd.to_datetime(df['data_admissao'], format='%Y-%m-%d', errors='coerce')
```
`errors='coerce'` transforma datas inválidas em `NaT`; conte e investigue essas ocorrências em vez de ignorá-las.


In [ ]:
import pandas as pd

df = pd.DataFrame({
    'nome': ['Ana', 'Bruno', 'Carlos', 'Ana', 'Eduarda'],
    'idade': [25, 30, None, 25, 29],
    'salario': [3000, 4000, 3500, 3000, None],
    'data_admissao': ['2020-01-01', '2019-05-20', '2021-03-15', '2020-01-01', '2022-07-10']
})
display(df.isna().sum())
# Exercício: teste imputação numa cópia e explique a decisão.
# Depois converta data_admissao para datetime e confira se apareceram NaT.

## Remoção de Duplicatas
---

Para remover registros duplicados, podemos utilizar o método `drop_duplicates()`:
```python
df = df.drop_duplicates()
```

Os parâmetros da função são os seguintes:

| Parâmetro      | Tipo         | Valor Padrão | Descrição                                                                     | Exemplo de Uso             |
| -------------- | ------------ | ------------ | ----------------------------------------------------------------------------- | -------------------------- |
| `subset`       | str ou lista | `None`       | Especifica as colunas a serem verificadas para duplicação.                    | `subset=['nome', 'email']` |
| `keep`         | str ou bool  | `'first'`    | Define qual duplicata manter: `'first'`, `'last'`, ou `False` (remove todas). | `keep='last'`              |
| `inplace`      | bool         | `False`      | Se `True`, modifica o DataFrame original sem retornar uma cópia.              | `inplace=True`             |
| `ignore_index` | bool         | `False`      | Se `True`, redefine os índices no DataFrame resultante. (Pandas ≥1.0.0)       | `ignore_index=True`        |

#### Exercício:

Exclua as duplicatas das linhas do DataFrame abaixo:

Remova apenas repetições comprovadamente indevidas. Sem uma chave ou regra de negócio, duas linhas com o mesmo nome podem ser registros legítimos. Neste exemplo, a linha inteira foi duplicada de propósito.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    'nome': ['Ana', 'Bruno', 'Carlos', 'Ana', 'Eduarda'],
    'idade': [25, 30, None, 25, 29],
    'salario': [3000, 4000, 3500, 3000, None],
    'data_admissao': ['2020-01-01', '2019-05-20', '2021-03-15', '2020-01-01', '2022-07-10']
})
print('Repetições integrais:', df.duplicated().sum())
# Exercício: remova somente a repetição integral e confira o tamanho.

# Exercício 2: Manipulação e limpeza de dados 🛠️

Use `player_salaries.csv`. Os atletas, clubes e valores desta edição são **fictícios** e os valores anuais estão em unidades monetárias de exemplo.

1. Inspecione tamanho, tipos e ausências. Conte e remova a repetição integral inserida intencionalmente.
2. Reproduza, em uma variável separada, `dropna()` sem `subset` e registre quantas linhas ele remove. Preserve uma cópia para comparar.
3. Para responder à pergunta, exija somente as colunas necessárias: `Player`, `Nation`, `Team`, `Age`, `Annual`. Uma posição ausente não deve excluir um atleta de uma comparação de idade e salário.
4. Filtre `Nation == 'BRA'` e `Age < 26`, ordene `Annual` em ordem decrescente e mostre nome, nacionalidade, clube e salário anual.
5. Qual atleta fictício aparece no topo? O resultado muda se todas as colunas forem exigidas como completas?
6. Explique as limitações: idade ou salário ausentes impedem avaliar alguns candidatos. A conclusão vale somente para os registros elegíveis com valores conhecidos e não comprova quem recebe mais numa liga real. Não some salários semanais e anuais nem compare valores de moedas/períodos diferentes.


In [ ]:
# Exercício 2: escreva aqui. A base está em BASE / 'player_salaries.csv'.

# Agrupamento e agregação 🗃️

O agrupamento e a agregação são duas formas úteis de descrever dados, resumir dados para análises, encontrar tendências por categoria e facilitar relatórios e visualizações.
  * **Agrupamento** é separar os dados em grupos com base em uma ou mais colunas.
  * **Agregação** é calcular algum resumo (média, soma, contagem, etc.) para cada grupo.

## Agrupamento
---

### O que é `groupby()`?

* Método que cria grupos a partir de uma coluna (ou conjunto de colunas).
* Depois, podemos aplicar funções de agregação para cada grupo.

#### Exemplo básico:

```python
import pandas as pd

# Dados de exemplo
data = {
    'categoria': ['A', 'A', 'B', 'B', 'C', 'C', 'C'],
    'valor': [10, 15, 10, 20, 5, 10, 15]
}
df = pd.DataFrame(data)

# Agrupar pela coluna 'categoria' e calcular a média da coluna 'valor'
df_agrupado = df.groupby('categoria')['valor'].mean()

print(df_agrupado)
```

**Resultado esperado:**

```
categoria
A    12.5
B    15.0
C    10.0
Name: valor, dtype: float64
```

Explicação:

* O pandas agrupa todas as linhas que têm a mesma categoria e calcula a média do valor dentro desse grupo.

## Agregação
---

### Funções de agregação comuns:

| Função    | O que faz                       | Exemplo                                    |
| --------- | ------------------------------- | ------------------------------------------ |
| `sum()`   | Soma os valores                 | `df.groupby('categoria')['valor'].sum()`   |
| `mean()`  | Calcula a média dos valores     | `df.groupby('categoria')['valor'].mean()`  |
| `count()` | Conta valores não ausentes da coluna     | `df.groupby('categoria')['valor'].count()` |
| `max()`   | Retorna o valor máximo do grupo | `df.groupby('categoria')['valor'].max()`   |
| `min()`   | Retorna o valor mínimo do grupo | `df.groupby('categoria')['valor'].min()`   |

Exemplo:

```python
print(df.groupby('categoria')['valor'].sum())
print(df.groupby('categoria')['valor'].count())
```

`groupby(...).size()` conta as linhas de cada grupo, incluindo linhas cuja coluna de salário esteja ausente. `count()` conta valores não ausentes da coluna selecionada. Por padrão, grupos com chave ausente são omitidos; use `dropna=False` se desejar mostrá-los. `sum(min_count=1)` preserva ausência quando um grupo não tem nenhum valor conhecido, evitando exibir zero como total observado.

## Tabelas Dinâmicas com `pivot_table()`

#### O que é uma tabela dinâmica?

* É uma forma de agrupar dados com mais flexibilidade, parecida com tabelas do Excel.
* Permite agregar valores com funções como média, soma, contagem, etc.
* Pode ter múltiplos índices e colunas.

### Exemplo:

```python
df_pivot = df.pivot_table(values='valor', index='categoria', aggfunc='mean')
print(df_pivot)
```

**Resultado:**

```
           valor
categoria       
A           12.5
B           15.0
C           10.0
```

Explicação:

* `values='valor'` diz qual coluna será agregada.
* `index='categoria'` diz que os grupos serão formados pela coluna 'categoria'.
* `aggfunc='mean'` escolhe a função agregadora (média).

---

#### Exemplo com categorias nas linhas, tipos nas colunas e uma agregação (soma):

```python
data = {
    'categoria': ['A', 'A', 'B', 'B', 'C', 'C', 'C'],
    'tipo': ['X', 'Y', 'X', 'Y', 'X', 'X', 'Y'],
    'valor': [10, 15, 10, 20, 5, 10, 15]
}
df2 = pd.DataFrame(data)

df_pivot2 = df2.pivot_table(values='valor', index='categoria', columns='tipo', aggfunc='sum', fill_value=0)
print(df_pivot2)
```

**Resultado:**

```
tipo       X   Y
categoria        
A         10  15
B         10  20
C         15  15
```

* Agora temos a soma dos valores agrupados por duas dimensões: `categoria` e `tipo`.

O parâmetro `aggfunc='mean'` exclui ausências do cálculo da média. `fill_value=0` preenche células da tabela final; use-o somente quando zero for uma representação adequada para aquela combinação.

# Visualização Básica com Pandas 🖥️

No pandas, é possível criar gráficos simples diretamente a partir dos DataFrames. Essas exibições gráficas são extremamente úteis para análises exploratórias rápidas.

## Tipos de Gráficos Suportados
---

O Pandas possui integração com o `matplotlib`, o que permite criar gráficos de forma simples usando métodos como:

* **Gráfico de Linhas** – Útil para sequências ordenadas, como séries temporais; a ordem do eixo precisa fazer sentido.

  ```python
  df.plot()
  ```

* **Gráfico de Barras** – Útil para comparar categorias.

  ```python
  df.plot.bar()
  ```

* **Histograma** – Excelente para entender a distribuição de uma variável numérica.

  ```python
  df['coluna'].hist()
  ```

## Personalização Básica
---

Você pode personalizar os gráficos com:

* **Título**: usando o parâmetro `title`
* **Rótulos dos eixos**: usando `plt.xlabel()` e `plt.ylabel()`

## Exemplo Prático

Suponha que temos um DataFrame com vendas por categoria:

```python
import pandas as pd
import matplotlib.pyplot as plt

# Exemplo de dados
dados = {
    'categoria': ['A', 'B', 'A', 'C', 'B', 'A'],
    'valor': [100, 200, 150, 300, 250, 50]
}
df = pd.DataFrame(dados)

# Agrupamento e plotagem
df.groupby('categoria')['valor'].sum().plot.bar(title='Soma por Categoria')

# Rótulos dos eixos
plt.xlabel('Categoria')
plt.ylabel('Valor Total')
plt.tight_layout()
plt.show()
```

**Saída esperada**: Um gráfico de barras com as categorias A, B e C no eixo X e a soma dos valores correspondentes no eixo Y.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Exemplo de dados
dados = {
    'categoria': ['A', 'B', 'A', 'C', 'B', 'A'],
    'valor': [100, 200, 150, 300, 250, 50]
}
df = pd.DataFrame(dados)

# Agrupamento e plotagem
df.groupby('categoria')['valor'].sum().plot.bar(title='Soma por Categoria')

# Rótulos dos eixos
plt.xlabel('Categoria')
plt.ylabel('Valor Total')
plt.tight_layout()
plt.show()

# Exercício 3: Agrupamento, Agregação e Visualização

Leia `dados_funcionarios.csv`, com pessoas e remunerações fictícias.

1. Remova somente a linha integral repetida, verificando a chave `ID`.
2. Agrupe por `Departamento` e calcule a média dos salários conhecidos. Mostre também total de pessoas (`size`) e número de salários conhecidos (`count`).
3. Faça uma tabela dinâmica com a soma dos salários conhecidos. Não apresente a soma como folha completa quando houver salários ausentes.
4. Crie um **gráfico de barras** da média dos salários conhecidos por departamento. Departamentos são categorias sem uma ordem contínua, por isso uma linha sugeriria uma continuidade inadequada.
5. Crie outro gráfico de barras com o número de pessoas por departamento e um histograma das idades conhecidas. Não preencha idades ausentes com zero.
6. Inclua título, unidades e rótulos dos eixos. Indique os valores ausentes relevantes e lembre que os dados são sintéticos.


In [ ]:
# Exercício 3: escreva aqui. A base está em BASE / 'dados_funcionarios.csv'.

# Referências e revisão

Documentação oficial consultada em 26/10/2026:

- [Estruturas de dados](https://pandas.pydata.org/docs/user_guide/dsintro.html)
- [Leitura CSV](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html)
- [Exportação CSV](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html)
- [Exportação Excel](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_excel.html)
- [ExcelWriter](https://pandas.pydata.org/docs/reference/api/pandas.ExcelWriter.html)
- [Copy-on-Write](https://pandas.pydata.org/pandas-docs/version/3.0/user_guide/copy_on_write.html)
- [Valores ausentes](https://pandas.pydata.org/docs/user_guide/missing_data.html)
- [Agrupamento](https://pandas.pydata.org/docs/user_guide/groupby.html)

Consulte `CORRECOES.md` para as mudanças em relação ao material apresentado. Revisão técnica e organização para publicação com apoio do OpenAI Codex. A versão histórica permanece preservada fora do repositório público.
